# Estudio Completo: Clasificación y Regresión

Análisis exhaustivo de algoritmos de Machine Learning incluyendo:
- EDA y Sanitización de Datos
- Clasificación: KNN, Decision Trees, Random Forest, XGBoost, AdaBoost
- Regresión: Linear, Lasso, Ridge, SVR, Tree-based models
- Benchmarking y comparación de modelos

## 1. Importar librerías y cargar datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, r2_score, mean_squared_error, mean_absolute_error
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Lasso, LassoCV, Ridge, RidgeCV
from sklearn.svm import SVR
from xgboost import XGBClassifier
import sys

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Agregar la ruta del proyecto
sys.path.append('c:/Users/dani1/source/repos/casoEstudio2')

print("✓ Librerías importadas exitosamente")

In [ ]:
# Cargar datasets
df_diabetes = pd.read_csv('diabetes_V2.csv', index_col=0)
df_potabilidad = pd.read_csv('potabilidad_V2.csv', index_col=0)

print("Dataset Diabetes:")
print(f"Shape: {df_diabetes.shape}")
print(f"Columnas: {list(df_diabetes.columns)}")
print(f"\nPrimeras filas:\n{df_diabetes.head()}")

print("\n" + "="*80)
print("\nDataset Potabilidad:")
print(f"Shape: {df_potabilidad.shape}")
print(f"Columnas: {list(df_potabilidad.columns)}")
print(f"\nPrimeras filas:\n{df_potabilidad.head()}")

## 2. EDA y Sanitización de Datos

In [ ]:
def analizar_dataset(df, nombre="Dataset"):
    """Realiza análisis completo del dataset"""
    print(f"\n{'='*80}")
    print(f"ANÁLISIS EDA: {nombre}")
    print(f"{'='*80}\n")
    
    # Información básica
    print(f"Shape: {df.shape[0]} filas × {df.shape[1]} columnas")
    print(f"\nTipos de datos:\n{df.dtypes}\n")
    
    # Valores nulos
    print(f"Valores Nulos:\n{df.isnull().sum()}\n")
    
    # Estadísticas descriptivas
    print(f"Estadísticas Descriptivas:\n{df.describe().T}\n")
    
    # Asimetría y Curtosis
    print(f"Asimetría (Skewness):\n{df.skew()}\n")
    
    return df

# Analizar datasets
df_diabetes_clean = analizar_dataset(df_diabetes, "Diabetes")
df_potabilidad_clean = analizar_dataset(df_potabilidad, "Potabilidad")

In [ ]:
# Sanitización: Eliminar nulos
df_diabetes_clean = df_diabetes.dropna()
df_potabilidad_clean = df_potabilidad.dropna()

print("Después de eliminar nulos:")
print(f"Diabetes: {df_diabetes_clean.shape}")
print(f"Potabilidad: {df_potabilidad_clean.shape}")

# Visualizar correlaciones
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(df_diabetes_clean.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[0], cbar_kws={'label': 'Correlación'})
axes[0].set_title('Correlaciones - Diabetes', fontsize=12, fontweight='bold')

sns.heatmap(df_potabilidad_clean.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1], cbar_kws={'label': 'Correlación'})
axes[1].set_title('Correlaciones - Potabilidad', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ EDA completado y datos sanitizados")

## 3. Preparación de datos para Clasificación

Usaremos el dataset de Diabetes (última columna como variable objetivo)

In [ ]:
# Preparar datos para clasificación (Diabetes)
# Asumiendo que la última columna es la variable objetivo
X = df_diabetes_clean.iloc[:, :-1]
y = df_diabetes_clean.iloc[:, -1]

# Normalizar features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42, stratify=y)

print(f"Datos preparados para clasificación:")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")
print(f"Distribución de clases en y_train: {y_train.value_counts().to_dict()}")

## 4. Análisis KNN - K-Nearest Neighbors

In [ ]:
# KNN - Configuración estándar y variaciones
resultados_knn = {}

# Parámetros a probar
k_values = [3, 5, 7, 9, 15, 21]
algorithms = ['auto', 'ball_tree', 'kd_tree', 'brute']

for k in k_values:
    for algo in algorithms:
        try:
            knn = KNeighborsClassifier(n_neighbors=k, algorithm=algo)
            knn.fit(X_train, y_train)
            y_pred = knn.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            resultados_knn[f"KNN(k={k}, algo={algo})"] = accuracy
        except:
            pass

# Ordenar resultados
resultados_knn_sorted = dict(sorted(resultados_knn.items(), key=lambda x: x[1], reverse=True))

print("="*80)
print("RESULTADOS KNN - Configuraciones")
print("="*80)
for modelo, acc in list(resultados_knn_sorted.items())[:10]:
    print(f"{modelo:40s} → Accuracy: {acc:.4f}")

# Mejor modelo KNN
mejor_modelo_knn = max(resultados_knn.items(), key=lambda x: x[1])
print(f"\n✓ MEJOR MODELO KNN: {mejor_modelo_knn[0]} con Accuracy: {mejor_modelo_knn[1]:.4f}")

In [ ]:
# Análisis detallado del mejor modelo KNN
mejor_k = int(mejor_modelo_knn[0].split('k=')[1].split(',')[0])
mejor_algo = mejor_modelo_knn[0].split('algo=')[1].rstrip(')')

knn_best = KNeighborsClassifier(n_neighbors=mejor_k, algorithm=mejor_algo)
knn_best.fit(X_train, y_train)
y_pred_knn = knn_best.predict(X_test)

print(f"\nANÁLISIS DETALLADO - MEJOR MODELO KNN (k={mejor_k}, algo={mejor_algo})")
print("="*80)
print(f"Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"\nReporte de Clasificación:\n{classification_report(y_test, y_pred_knn)}")
print(f"\nMatriz de Confusión:\n{confusion_matrix(y_test, y_pred_knn)}")

# Visualizar matriz de confusión
cm = confusion_matrix(y_test, y_pred_knn)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Matriz de Confusión - KNN (k={mejor_k})', fontweight='bold')
plt.ylabel('Valor Real')
plt.xlabel('Valor Predicho')
plt.show()

## 5. Análisis Decision Trees

In [ ]:
# Decision Trees - Configuraciones estándar y variaciones
resultados_dt = {}

max_depths = [3, 5, 10, 15, None]
min_samples_splits = [2, 5, 10]
criteria = ['gini', 'entropy']

for depth in max_depths:
    for min_samp in min_samples_splits:
        for criterion in criteria:
            dt = DecisionTreeClassifier(max_depth=depth, min_samples_split=min_samp, criterion=criterion, random_state=42)
            dt.fit(X_train, y_train)
            y_pred = dt.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            resultados_dt[f"DT(depth={depth}, min_samp={min_samp}, criterion={criterion})"] = accuracy

# Ordenar resultados
resultados_dt_sorted = dict(sorted(resultados_dt.items(), key=lambda x: x[1], reverse=True))

print("="*80)
print("RESULTADOS DECISION TREES - Top 10 Configuraciones")
print("="*80)
for modelo, acc in list(resultados_dt_sorted.items())[:10]:
    print(f"{modelo:60s} → Accuracy: {acc:.4f}")

# Mejor modelo Decision Tree
mejor_modelo_dt = max(resultados_dt.items(), key=lambda x: x[1])
print(f"\n✓ MEJOR MODELO DT: {mejor_modelo_dt[0]} con Accuracy: {mejor_modelo_dt[1]:.4f}")

In [ ]:
# Análisis detallado del mejor Decision Tree
params_dt = mejor_modelo_dt[0]
depth_val = int(params_dt.split('depth=')[1].split(',')[0]) if params_dt.split('depth=')[1].split(',')[0] != 'None' else None
min_samp_val = int(params_dt.split('min_samp=')[1].split(',')[0])
criterion_val = params_dt.split('criterion=')[1].rstrip(')')

dt_best = DecisionTreeClassifier(max_depth=depth_val, min_samples_split=min_samp_val, criterion=criterion_val, random_state=42)
dt_best.fit(X_train, y_train)
y_pred_dt = dt_best.predict(X_test)

print(f"\nANÁLISIS DETALLADO - MEJOR DECISION TREE")
print("="*80)
print(f"Parámetros: max_depth={depth_val}, min_samples_split={min_samp_val}, criterion={criterion_val}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"\nReporte de Clasificación:\n{classification_report(y_test, y_pred_dt)}")

# Feature importance
importances = dt_best.feature_importances_
feature_names = X_train.columns
indices = np.argsort(importances)[-10:]

plt.figure(figsize=(10, 6))
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), feature_names[indices])
plt.xlabel('Importancia')
plt.title('Top 10 Features - Decision Tree', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Análisis Random Forest

In [ ]:
# Random Forest - Configuraciones estándar y variaciones
resultados_rf = {}

n_estimators_list = [50, 100, 200, 300]
max_depths_rf = [5, 10, 15, 20, None]
min_samples_splits_rf = [2, 5, 10]

for n_est in n_estimators_list:
    for depth in max_depths_rf:
        for min_samp in min_samples_splits_rf:
            rf = RandomForestClassifier(n_estimators=n_est, max_depth=depth, min_samples_split=min_samp, random_state=42, n_jobs=-1)
            rf.fit(X_train, y_train)
            y_pred = rf.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            resultados_rf[f"RF(n_est={n_est}, depth={depth}, min_samp={min_samp})"] = accuracy

# Ordenar resultados
resultados_rf_sorted = dict(sorted(resultados_rf.items(), key=lambda x: x[1], reverse=True))

print("="*80)
print("RESULTADOS RANDOM FOREST - Top 10 Configuraciones")
print("="*80)
for modelo, acc in list(resultados_rf_sorted.items())[:10]:
    print(f"{modelo:65s} → Accuracy: {acc:.4f}")

# Mejor modelo Random Forest
mejor_modelo_rf = max(resultados_rf.items(), key=lambda x: x[1])
print(f"\n✓ MEJOR MODELO RF: Accuracy: {mejor_modelo_rf[1]:.4f}")

In [ ]:
# Análisis detallado del mejor Random Forest
params_rf = mejor_modelo_rf[0]
n_est_val = int(params_rf.split('n_est=')[1].split(',')[0])
depth_rf_val = int(params_rf.split('depth=')[1].split(',')[0]) if params_rf.split('depth=')[1].split(',')[0] != 'None' else None
min_samp_rf_val = int(params_rf.split('min_samp=')[1].rstrip(')'))

rf_best = RandomForestClassifier(n_estimators=n_est_val, max_depth=depth_rf_val, min_samples_split=min_samp_rf_val, random_state=42, n_jobs=-1)
rf_best.fit(X_train, y_train)
y_pred_rf = rf_best.predict(X_test)

print(f"\nANÁLISIS DETALLADO - MEJOR RANDOM FOREST")
print("="*80)
print(f"Parámetros: n_estimators={n_est_val}, max_depth={depth_rf_val}, min_samples_split={min_samp_rf_val}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"\nReporte de Clasificación:\n{classification_report(y_test, y_pred_rf)}")

# Feature importance
importances_rf = rf_best.feature_importances_
indices_rf = np.argsort(importances_rf)[-10:]

plt.figure(figsize=(10, 6))
plt.barh(range(len(indices_rf)), importances_rf[indices_rf])
plt.yticks(range(len(indices_rf)), feature_names[indices_rf])
plt.xlabel('Importancia')
plt.title('Top 10 Features - Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Análisis XGBoost y AdaBoost

In [ ]:
# XGBoost - Configuraciones
resultados_xgb = {}

n_estimators_xgb = [50, 100, 150, 200]
max_depths_xgb = [3, 5, 7, 10]
learning_rates = [0.01, 0.05, 0.1, 0.2]

print("Entrenando modelos XGBoost... (esto puede tomar un tiempo)")
for n_est in n_estimators_xgb[:2]:  # Limitado para no tardar mucho
    for depth in max_depths_xgb[:2]:
        for lr in learning_rates:
            try:
                xgb = XGBClassifier(n_estimators=n_est, max_depth=depth, learning_rate=lr, random_state=42, verbosity=0)
                xgb.fit(X_train, y_train)
                y_pred = xgb.predict(X_test)
                accuracy = accuracy_score(y_test, y_pred)
                resultados_xgb[f"XGB(n_est={n_est}, depth={depth}, lr={lr})"] = accuracy
            except:
                pass

# AdaBoost
resultados_ada = {}
n_estimators_ada = [50, 100, 150, 200]
learning_rates_ada = [0.5, 0.8, 1.0, 1.2]

for n_est in n_estimators_ada:
    for lr in learning_rates_ada:
        ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=3), n_estimators=n_est, learning_rate=lr, random_state=42)
        ada.fit(X_train, y_train)
        y_pred = ada.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        resultados_ada[f"ADA(n_est={n_est}, lr={lr})"] = accuracy

# Mostrar resultados
print("\n" + "="*80)
print("RESULTADOS XGBOOST - Top 5")
print("="*80)
xgb_sorted = dict(sorted(resultados_xgb.items(), key=lambda x: x[1], reverse=True))
for modelo, acc in list(xgb_sorted.items())[:5]:
    print(f"{modelo:50s} → Accuracy: {acc:.4f}")

print("\n" + "="*80)
print("RESULTADOS ADABOOST - Top 5")
print("="*80)
ada_sorted = dict(sorted(resultados_ada.items(), key=lambda x: x[1], reverse=True))
for modelo, acc in list(ada_sorted.items())[:5]:
    print(f"{modelo:40s} → Accuracy: {acc:.4f}")

mejor_modelo_xgb = max(resultados_xgb.items(), key=lambda x: x[1])
mejor_modelo_ada = max(resultados_ada.items(), key=lambda x: x[1])

print(f"\n✓ MEJOR XGBoost: Accuracy {mejor_modelo_xgb[1]:.4f}")
print(f"✓ MEJOR AdaBoost: Accuracy {mejor_modelo_ada[1]:.4f}")

## 8. MODELOS DE REGRESIÓN

### Preparación de datos para Regresión (Dataset Potabilidad)

In [ ]:
# Preparar datos para regresión (Potabilidad)
# Usaremos la primera columna numérica como objetivo de regresión
X_reg = df_potabilidad_clean.iloc[:, 1:]  # Todas las columnas excepto la primera
y_reg = df_potabilidad_clean.iloc[:, 0]   # Primera columna como objetivo

print(f"Dataset Potabilidad para Regresión:")
print(f"X_reg shape: {X_reg.shape}, y_reg shape: {y_reg.shape}")

# Normalizar
scaler_reg = StandardScaler()
X_reg_scaled = scaler_reg.fit_transform(X_reg)
X_reg_scaled = pd.DataFrame(X_reg_scaled, columns=X_reg.columns)

# Dividir datos
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg_scaled, y_reg, test_size=0.25, random_state=42
)

print(f"Datos preparados:")
print(f"X_train_reg: {X_train_reg.shape}, X_test_reg: {X_test_reg.shape}")
print(f"Estadísticas de y_reg: min={y_reg.min():.2f}, max={y_reg.max():.2f}, media={y_reg.mean():.2f}")

In [ ]:
def evaluar_regresion(y_true, y_pred, nombre_modelo=""):
    """Evalúa un modelo de regresión"""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}

# Almacenar resultados de regresión
resultados_reg = {}

print("\n" + "="*80)
print("MODELOS DE REGRESIÓN")
print("="*80)

### 8.1 Linear Regression (Simple y Múltiple)

In [ ]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train_reg, y_train_reg)
y_pred_lr = lr.predict(X_test_reg)

metricas_lr = evaluar_regresion(y_test_reg, y_pred_lr)
resultados_reg["Linear Regression"] = metricas_lr

print("\nLinear Regression:")
print(f"  R²: {metricas_lr['R2']:.4f}")
print(f"  RMSE: {metricas_lr['RMSE']:.4f}")
print(f"  MAE: {metricas_lr['MAE']:.4f}")
print(f"  Coeficientes: {lr.coef_[:5]} (primeros 5)")  # Mostrar primeros 5 coef

### 8.2 Lasso, LassoCV, Ridge, RidgeCV

In [ ]:
# Lasso
lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train_reg, y_train_reg)
y_pred_lasso = lasso.predict(X_test_reg)
metricas_lasso = evaluar_regresion(y_test_reg, y_pred_lasso)
resultados_reg["Lasso"] = metricas_lasso

# LassoCV
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso_cv.fit(X_train_reg, y_train_reg)
y_pred_lasso_cv = lasso_cv.predict(X_test_reg)
metricas_lasso_cv = evaluar_regresion(y_test_reg, y_pred_lasso_cv)
resultados_reg["LassoCV"] = metricas_lasso_cv

# Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_reg, y_train_reg)
y_pred_ridge = ridge.predict(X_test_reg)
metricas_ridge = evaluar_regresion(y_test_reg, y_pred_ridge)
resultados_reg["Ridge"] = metricas_ridge

# RidgeCV
ridge_cv = RidgeCV(alphas=np.logspace(-2, 2, 10), cv=5)
ridge_cv.fit(X_train_reg, y_train_reg)
y_pred_ridge_cv = ridge_cv.predict(X_test_reg)
metricas_ridge_cv = evaluar_regresion(y_test_reg, y_pred_ridge_cv)
resultados_reg["RidgeCV"] = metricas_ridge_cv

print("\nResultados Regresión Regularizada:")
for nombre, met in [("Lasso", metricas_lasso), ("LassoCV", metricas_lasso_cv), ("Ridge", metricas_ridge), ("RidgeCV", metricas_ridge_cv)]:
    print(f"{nombre:8s} - R²: {met['R2']:.4f}, RMSE: {met['RMSE']:.4f}, MAE: {met['MAE']:.4f}")
print(f"Alpha óptimo LassoCV: {lasso_cv.alpha_:.4f}")
print(f"Alpha óptimo RidgeCV: {ridge_cv.alpha_:.4f}")

### 8.3 Support Vector Regression (SVR)

In [ ]:
# SVR con diferentes kernels
kernels = ['linear', 'rbf', 'poly']
for kernel in kernels:
    svr = SVR(kernel=kernel, C=100, epsilon=0.1)
    svr.fit(X_train_reg, y_train_reg)
    y_pred_svr = svr.predict(X_test_reg)
    metricas = evaluar_regresion(y_test_reg, y_pred_svr)
    resultados_reg[f"SVR_{kernel}"] = metricas
    print(f"SVR ({kernel}) - R²: {metricas['R2']:.4f}, RMSE: {metricas['RMSE']:.4f}, MAE: {metricas['MAE']:.4f}")


### 8.4 Decision Tree y Random Forest Regression

In [ ]:
# Decision Tree Regressor
for depth in [3, 5, 8, None]:
    dt_reg = DecisionTreeRegressor(max_depth=depth, random_state=42)
    dt_reg.fit(X_train_reg, y_train_reg)
    y_pred_dt_reg = dt_reg.predict(X_test_reg)
    metricas = evaluar_regresion(y_test_reg, y_pred_dt_reg)
    resultados_reg[f"DT_Reg(depth={depth})"] = metricas
    print(f"DecisionTreeRegressor(depth={depth}) - R²: {metricas['R2']:.4f}, RMSE: {metricas['RMSE']:.4f}")

# Random Forest Regressor
for n_est in [50, 100, 200]:
    rf_reg = RandomForestRegressor(n_estimators=n_est, max_depth=10, random_state=42, n_jobs=-1)
    rf_reg.fit(X_train_reg, y_train_reg)
    y_pred_rf_reg = rf_reg.predict(X_test_reg)
    metricas = evaluar_regresion(y_test_reg, y_pred_rf_reg)
    resultados_reg[f"RF_Reg(n_est={n_est})"] = metricas
    print(f"RandomForestRegressor(n_estimators={n_est}) - R²: {metricas['R2']:.4f}, RMSE: {metricas['RMSE']:.4f}")


## 9. Comparación y Benchmarking de Modelos de Regresión

In [ ]:
# Crear DataFrame comparativo para regresión
comparacion_reg = pd.DataFrame({
    nombre: {"R2": met["R2"], "RMSE": met["RMSE"], "MAE": met["MAE"]}
    for nombre, met in resultados_reg.items()
}).T

print(comparacion_reg.sort_values(by='R2', ascending=False).to_string())

# Visualización comparativa
comparacion_reg[['R2', 'RMSE']].plot(kind='bar', secondary_y='RMSE', figsize=(14, 7))
plt.title('Benchmarking Modelos de Regresión')
plt.tight_layout()
plt.show()

## 10. Conclusiones

- Se validaron los datasets y se limpiaron datos faltantes.
- Se evaluaron KNN, Decision Tree, Random Forest, XGBoost y AdaBoost para clasificación.
- Se entrenaron modelos de regresión con Linear Regression, Lasso, LassoCV, Ridge, RidgeCV, SVR, DecisionTreeRegressor y RandomForestRegressor.
- Se generó un benchmarking para comparar resultados y seleccionar los mejores modelos.

> Use este notebook como base para documentar el estudio de regresiones y la selección del mejor modelo.

In [ ]:
# Lasso
lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train_reg, y_train_reg)
y_pred_lasso = lasso.predict(X_test_reg)
metricas_lasso = evaluar_regresion(y_test_reg, y_pred_lasso)
resultados_reg["Lasso (α=0.1)"] = metricas_lasso

# LassoCV (busca el mejor alpha)
lassocv = LassoCV(cv=5, max_iter=10000, random_state=42)
lassocv.fit(X_train_reg, y_train_reg)
y_pred_lassocv = lassocv.predict(X_test_reg)
metricas_lassocv = evaluar_regresion(y_test_reg, y_pred_lassocv)
resultados_reg[f"LassoCV (α={lassocv.alpha_:.4f})"] = metricas_lassocv

# Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_reg, y_train_reg)
y_pred_ridge = ridge.predict(X_test_reg)
metricas_ridge = evaluar_regresion(y_test_reg, y_pred_ridge)
resultados_reg["Ridge (α=1.0)"] = metricas_ridge

# RidgeCV
ridgecv = RidgeCV(alphas=np.logspace(-2, 10, 13), cv=5)
ridgecv.fit(X_train_reg, y_train_reg)
y_pred_ridgecv = ridgecv.predict(X_test_reg)
metricas_ridgecv = evaluar_regresion(y_test_reg, y_pred_ridgecv)
resultados_reg[f"RidgeCV (α={ridgecv.alpha_:.4f})"] = metricas_ridgecv

print("\nLasso & Ridge Regression:")
print(f"  Lasso (α=0.1): R² = {metricas_lasso['R2']:.4f}, RMSE = {metricas_lasso['RMSE']:.4f}")
print(f"  LassoCV (α={lassocv.alpha_:.4f}): R² = {metricas_lassocv['R2']:.4f}, RMSE = {metricas_lassocv['RMSE']:.4f}")
print(f"  Ridge (α=1.0): R² = {metricas_ridge['R2']:.4f}, RMSE = {metricas_ridge['RMSE']:.4f}")
print(f"  RidgeCV (α={ridgecv.alpha_:.4f}): R² = {metricas_ridgecv['R2']:.4f}, RMSE = {metricas_ridgecv['RMSE']:.4f}")